# Recurrent neural networks

## 1. The intuition: why does a model need memory?

Notebook 01 ended with two gaps in the embedding lookup:

- It is **static**: "bear" gets the same vector in every sentence.
- It has **no word order**: "The teddy bear" and "bear teddy The" give the same rows.

An RNN is the first fix. It reads the sentence one token at a time and carries a memory along.

> A cute teddy bear is reading.

At each step, the RNN reads one token and updates its **hidden state**, a running summary of everything it has read so far:

![An RNN reading "A cute teddy bear is reading" one token at a time](images/rnn_unrolled.svg)

- $h_t$ is the memory after reading $t$ tokens.
- $h_t$ is computed from the current token $x_t$ and the previous memory $h_{t-1}$.
- The same RNN cell, with the same weights, is reused at every step.

## 2. The mathematics: how does an RNN update its memory?

We keep the sentence "A cute teddy bear is reading". Suppose we have:

- Sequence length $T=6$
- Embedding dimension $d=4$ (as in notebook 01)
- Hidden size $H=3$ (the length of the memory vector)

#### 2.1 The pieces

| Symbol | Name | Intuition | Example |
|---|---|---|---|
| $x_t$ | Input embedding | • The embedding of the token at step $t$<br>• Looked up from $E$, as in notebook 01 | $x_3$ is the row for "teddy": $d = 4$ numbers |
| $h_{t-1}$ | Previous hidden state | • The memory before reading token $t$ | $h_2$ summarizes "A cute" |
| $h_t$ | New hidden state | • The memory after reading token $t$<br>• Same length at every step | $h_3$ summarizes "A cute teddy": $H = 3$ numbers |
| $W_{xh}$ | Input weights | • How the new token enters the memory<br>• Learned, reused at every step | Shape $H \times d = 3 \times 4$ |
| $W_{hh}$ | Memory weights | • How the old memory carries over<br>• Learned, reused at every step | Shape $H \times H = 3 \times 3$ |
| $b$ | Bias | • A learned constant added before tanh<br>• Shifts each memory number up or down | Length $H = 3$ |
| $\tanh$ | Squashing function | • Keeps every number between $-1$ and $1$ | $\tanh(0.80) \approx 0.66$ |

#### 2.2 One step

Every step does the same four things:

1. Multiply the token's embedding by $W_{xh}$, and the old memory by $W_{hh}$.
2. Add both results and a bias $b$.
3. Squash the sum with $\tanh$. The result is the new memory $h_t$.
4. Hand $h_t$ to the next step, where it becomes the old memory.

The weights are learned and shared by every step.

#### 2.3 Step 1 with numbers

Here are small made-up weights, shared by every step, for the first two steps of the sentence:

$$W_{xh} = \begin{bmatrix} 2 & 0 & 2 & 0 \\ 0 & 1 & 0 & 2 \\ 1 & 1 & -1 & 0 \end{bmatrix}, \quad W_{hh} = \begin{bmatrix} 1 & 0 & 0 \\ 0 & 1 & 1 \\ 1 & 0 & 1 \end{bmatrix}, \quad b = \begin{bmatrix} 0 \\ 0.1 \\ -0.1 \end{bmatrix}$$

**What each matrix does**

- $W_{xh}$ **reads the word.** Row $i$ says which of the word's 4 numbers feed memory number $i$. Row 1, $[2, 0, 2, 0]$, listens to numbers 1 and 3 and ignores 2 and 4.
- $W_{hh}$ **keeps the old memory.** Row $i$ says which of the old 3 memory numbers feed new memory number $i$. Row 1, $[1, 0, 0]$, just keeps old number 1.
- A big weight means "pay attention", a zero means "ignore", and a negative weight pushes the other way (row 3 of $W_{xh}$ subtracts the third number).
- If $W_{hh}$ were all zeros, the memory would forget the past at every step. If $W_{xh}$ were all zeros, it would ignore the words.

All numbers are rounded to 2 decimals at each step.

**How does $W_{xh}$ multiply $x_1$?**

![W_xh (3 by 4) times x_1 (4 by 1) gives 3 numbers](images/rnn_matmul.svg)

- $x_1$ is the embedding of "A": a column of $d = 4$ numbers.
- The inner sizes match: $(3 \times 4)(4 \times 1) = (3 \times 1)$.
- Each row of $W_{xh}$ times $x_1$, added up, gives one number. Row 1 gives $+0.80$.
- $W_{hh} h_{t-1}$ works the same way.

![Step 1: reading "A", starting from a memory of zeros](images/rnn_step1.svg)

- $h_0$ is all zeros, so $W_{hh} h_0$ adds nothing.
- $\tanh$ squashes the sum: $0.80$ becomes $0.66$.
- $h_1$ is the memory after reading "A".

**What does $\tanh$ do?**

- It squashes any number into the range $-1$ to $1$.
- Near 0 it barely changes the number. Far from 0, the curve flattens out near $\pm 1$.
- The two orange points are from step 1: $\tanh(0.80) = 0.66$ and $\tanh(-0.30) = -0.29$.

![The tanh curve, with two numbers from step 1](images/tanh_curve.svg)

#### 2.4 Step 2

![Step 2: reading "cute", starting from h1](images/rnn_step2.svg)

- $h_1$ is now the old memory, so $W_{hh} h_1$ is no longer zero. The memory of "A" flows into step 2.
- $h_2$ depends on both "A" and "cute".
- Step 3 would repeat this with $h_2$ and the embedding of "teddy".

#### 2.5 The same steps, as nodes

![All the nodes of step 1 and step 2, at a glance](images/rnn_nodes_overview.svg)

- Each node is a number: the word's 4 numbers, and the memory's 3 numbers at each step.
- **Amber lines:** $W_{xh}$ brings the word into the memory.
- **Purple lines:** $W_{hh}$ carries the old memory forward. In step 1 the old memory is all zeros, so these lines are faint.
- $h_1$ is the output of step 1 and the old memory of step 2.
- A dashed line is a negative weight. No line means the weight is 0.

Now one memory number at a time, with the exact numbers:

![Step 1 drawn as nodes and weights](images/rnn_nodes_step1.svg)

![Step 2 drawn as nodes and weights](images/rnn_nodes_step2.svg)

- Each memory number is built from three parts: a **word part** (through $W_{xh}$), a **memory part** (through $W_{hh}$) and a bias.
- Every input appears in every band. A grey $\times 0$ means the weight is 0: that input is multiplied by 0 and adds nothing.
- Step 1, memory number 1: the word part is $2 \times 0.10 + 2 \times 0.30 = 0.80$. The memory part is 0, because $h_0$ is all zeros. Then $\tanh(0.80) = 0.66$.
- Step 2, memory number 2: the word part is $+1.30$ and the memory part is $-0.39$. The old memory pulls this number down a little. Then $\tanh(1.01) = 0.77$.
- So $W_{xh}$ decides how the word shapes the memory, and $W_{hh}$ decides how the old memory carries on.

#### 2.6 The update rule

$$\boxed{h_t = \tanh(W_{xh}\, x_t + W_{hh}\, h_{t-1} + b)}$$

In words: new memory $= f(\text{current token}, \text{old memory})$. The first step starts from $h_0 = 0$.

**Key shapes**

| Piece | Shape | Why |
|---|---|---|
| $W_{xh}\, x_t$ | $(3 \times 4)(4 \times 1) = 3 \times 1$ | Turns a token embedding of length $d$ into length $H$ |
| $W_{hh}\, h_{t-1}$ | $(3 \times 3)(3 \times 1) = 3 \times 1$ | Turns the old memory of length $H$ into length $H$ |
| $h_t$ | $3 \times 1$ | Both products have length $H$, so they can be added |

The memory always has length $H$, whether the RNN has read 6 tokens or 600. A long sequence has to be squeezed into a fixed-size vector.

## 3. Reflection: why do RNNs struggle?

#### 3.1 Slow

![Six RNN steps run one after another, while independent steps could run together](images/rnn_sequential.svg)

- During training, all 6 tokens are known from the start.
- But $h_t$ needs $h_{t-1}$, so step 3 cannot start until step 2 is done.
- The steps run one after another: 6 tokens take 6 time slots, and 600 tokens take 600.
- If the steps did not depend on each other, all of them could run at the same time.

#### 3.2 Forgets

- To learn that "A" mattered, the training signal from the loss has to travel back to step 1.
- On the way, it is multiplied once per step.

![The training signal shrinks as it travels back through the steps](images/rnn_vanishing.svg)

- The factor depends on the weights $W_{hh}$ and the slope of $\tanh$. 0.8 and 1.2 are made-up examples.
- Each factor is below 1 (here 0.8), so the signal shrinks: $0.33$ after 5 steps, $0.01$ after 20.
- The early words get almost no signal, so the RNN struggles to learn from them. This is the **vanishing gradient**.
- If each factor is above 1 (say 1.2), the signal grows instead: $2.5$ after 5 steps, $38$ after 20. This is the **exploding gradient**.
- Where $\tanh$ is flat, near $\pm 1$, its slope is close to 0. That shrinks the signal even more.

#### 3.3 LSTM: a fix, with a limit

- An RNN rewrites its whole memory at every step. An LSTM edits it instead: erase a little, write a little.
- For this, the LSTM (1997) adds a second, long-term memory $c_t$.
- Three gates do the editing. A gate is a number between 0 and 1 for each memory number: 0 blocks, 1 lets through.

![The three LSTM gates, with numbers: forget, input and output](images/lstm_numbers.svg)

- **Forget gate:** how much of each old memory number to keep.
- **Input gate:** how much new information to write.
- **Output gate:** how much of each memory number to show as $h_t$.

**How this helps the vanishing gradient**

- **RNN:** rewrites its memory at every step, like retelling a story. The beginning gets lost.
- **LSTM:** edits its memory, like a notebook: new memory = forget gate $\times$ old memory + input gate $\times$ new information.
- Forget gate 1 and input gate 0: the new note equals the old note. A change at step 1 shows up unchanged at step 20, so the training signal can travel back along it at full strength.
- A gate is rarely exactly 1. With 0.95, 95% of the note survives each step, so $0.36$ of the signal is left after 20 steps, against $0.01$ for the RNN.
- The gates are learned, so the LSTM learns what to keep.

![RNN vs LSTM: how much training signal reaches each step](images/lstm_vs_rnn_signal.svg)

**The limit**

- It is still sequential: step $t$ needs step $t-1$, so it is still slow.
- The memory still has a fixed size.

#### 3.4 Summary

| Method | Pros | Cons |
|---|---|---|
| Static embeddings (e.g. word2vec) | • Very simple, yet powerful<br>• Intuitive embeddings | • Word order does not count<br>• Embeddings are not context aware |
| RNN, LSTM | • Word order matters<br>• State of the art at the time | • Vanishing gradient problem (3.2). The LSTM helps (3.3)<br>• Slow computations (3.1) |

- RNNs fix the two gaps of static embeddings: they use word order and context.
- They bring two new problems: they forget and they are slow.
- Attention is built to fix both (see the Attention notebook).

## 4. In PyTorch: what does `nn.RNN` do, and not do?

**Goal:** check in code what an RNN does, what it cannot do, and whether an LSTM helps.

- **Does:** follow our formula (4.1), and use context (4.2)
- **Cannot:** run its steps in parallel (4.3), or keep the first word over a long sequence (4.4)
- **LSTM:** does it keep the first word better? (4.5)

**Setup**

- The first two steps of "A cute teddy bear is reading": "A" and "cute"
- The same made-up weights as in 2.3
- 4.3 to 4.5 use longer sequences and random weights, as each of them says
- `nn.RNN` has two biases, one for the input and one for the memory. We use one, so we set the second to zero

In [1]:
import torch
import torch.nn as nn

torch.set_printoptions(precision=2, sci_mode=False)

# The weights from 2.3, shared by every step
W_xh = torch.tensor([[2., 0., 2., 0.],
                     [0., 1., 0., 2.],
                     [1., 1., -1., 0.]])   # (H, d) = (3, 4)
W_hh = torch.tensor([[1., 0., 0.],
                     [0., 1., 1.],
                     [1., 0., 1.]])        # (H, H) = (3, 3)
b = torch.tensor([0.0, 0.1, -0.1])         # (H,) = (3,)

# The embeddings of "A" and "cute"
x1 = torch.tensor([0.1, 0.0, 0.3, -0.1])
x2 = torch.tensor([0.2, 0.5, -0.1, 0.4])

#### 4.1 It does: it is our formula

- First, our formula $h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b)$ as a plain loop.
- Then `nn.RNN` with the same weights. Do they agree?

In [2]:
# Our formula, step by step
h = torch.zeros(3)                      # h_0 starts at zeros
manual = []
for x in (x1, x2):
    h = torch.tanh(W_xh @ x + W_hh @ h + b)
    manual.append(h)
manual = torch.stack(manual)            # (T, H)

# The same thing with nn.RNN, with our weights copied in
rnn = nn.RNN(input_size=4, hidden_size=3, batch_first=True)
with torch.no_grad():
    rnn.weight_ih_l0.copy_(W_xh)
    rnn.weight_hh_l0.copy_(W_hh)
    rnn.bias_ih_l0.copy_(b)
    rnn.bias_hh_l0.zero_()              # nn.RNN has two biases, we use one

X = torch.stack([x1, x2]).unsqueeze(0)  # (B, T, d) = (1, 2, 4)
output, h_n = rnn(X)

print("Our loop, h1 and h2:")
print(manual)
print("\nnn.RNN output, h1 and h2:")
print(output[0].detach())               # .detach() hides PyTorch's gradient bookkeeping
print("\nSame result?", torch.allclose(manual, output[0], atol=1e-6))
print("\nShape of X (B, T, d):      ", X.shape)
print("Shape of output (B, T, H): ", output.shape)
print("Shape of h_n (1, B, H):    ", h_n.shape)
print("h_n equals the last output?", torch.allclose(h_n[0], output[:, -1]))

Our loop, h1 and h2:
tensor([[ 0.66, -0.10, -0.29],
        [ 0.70,  0.77,  0.79]])

nn.RNN output, h1 and h2:
tensor([[ 0.66, -0.10, -0.29],
        [ 0.70,  0.77,  0.79]])

Same result? True

Shape of X (B, T, d):       torch.Size([1, 2, 4])
Shape of output (B, T, H):  torch.Size([1, 2, 3])
Shape of h_n (1, B, H):     torch.Size([1, 1, 3])
h_n equals the last output? True


| Letter | Meaning | In our code |
|---|---|---|
| $B$ | **Batch size**: how many sentences go through at the same time | 1 (one sentence) |
| $T$ | **Sequence length**: how many tokens in each sentence | 2 ("A cute") |
| $d$ | **Embedding dimension**: how many numbers describe each token | 4 |
| $H$ | **Hidden size**: how many numbers in the memory | 3 |

- Our loop and `nn.RNN` agree. $h_1$ and $h_2$ are the numbers from the pictures in 2.3 and 2.4.
- `output` has shape $(B, T, H)$: one hidden state for every step.
- `h_n` is the last hidden state. Its first dimension is the number of layers, here 1.
- `batch_first=True` puts the batch first. PyTorch's default order is $(T, B, d)$.

#### 4.2 It does: it uses context

- In notebook 01, the lookup gave "bear" the same vector in every sentence.
- Compare "teddy bear" and "grizzly bear". Does the hidden state at "bear" change?

In [3]:
# Embeddings (d = 4). teddy and bear are from notebook 01, grizzly is made up
teddy   = torch.tensor([+0.7, +0.1, -0.3, +0.8])
grizzly = torch.tensor([-0.5, +0.6, +0.5, -0.3])
bear    = torch.tensor([-0.1, +0.9, +0.4, +0.2])

teddy_bear   = torch.stack([teddy, bear]).unsqueeze(0)    # "teddy bear"
grizzly_bear = torch.stack([grizzly, bear]).unsqueeze(0)  # "grizzly bear"

out_teddy, _ = rnn(teddy_bear)
out_grizzly, _ = rnn(grizzly_bear)

print('Embedding of "bear" is the same in both?', torch.equal(teddy_bear[0, 1], grizzly_bear[0, 1]))
print('\nHidden state at "bear" after "teddy":  ', out_teddy[0, 1].detach())
print('Hidden state at "bear" after "grizzly":', out_grizzly[0, 1].detach())
print('\nSame hidden state?', torch.allclose(out_teddy[0, 1], out_grizzly[0, 1]))

Embedding of "bear" is the same in both? True

Hidden state at "bear" after "teddy":   tensor([0.85, 1.00, 0.94])
Hidden state at "bear" after "grizzly": tensor([ 0.54,  0.78, -0.16])

Same hidden state? False


- The embedding of "bear" is identical in both.
- The hidden state at "bear" is different, because it also carries the memory of "teddy" or "grizzly".
- This is the context the lookup could not give.

#### 4.3 It cannot: run its steps in parallel

- Section 3.1 said step $t$ needs step $t-1$.
- Check it on all six tokens: if we change one token, which hidden states change?

In [4]:
def run_rnn(X):                          # X: (T, d), returns the hidden states (T, H)
    h = torch.zeros(3)
    out = []
    for x in X:
        h = torch.tanh(W_xh @ x + W_hh @ h + b)
        out.append(h)
    return torch.stack(out)

torch.manual_seed(0)
X6 = torch.randn(6, 4)                   # six made-up embeddings: A cute teddy bear is reading
H_all = run_rnn(X6)

def which_change(X_new):                 # for each step: did its hidden state change?
    same = torch.isclose(run_rnn(X_new), H_all).all(dim=1)
    return (~same).tolist()

first = X6.clone(); first[0] += 1.0      # change the first token
last = X6.clone(); last[5] += 1.0        # change the last token

print("Change the first token. Hidden states that change:")
print(which_change(first))
print("\nChange the last token. Hidden states that change:")
print(which_change(last))

# The input part W_xh x_t does not need h, so all six steps fit in one matrix product
print("\nW_xh x_t for all 6 steps at once:", (X6 @ W_xh.T).shape)

Change the first token. Hidden states that change:
[True, True, True, True, True, True]

Change the last token. Hidden states that change:
[False, False, False, False, False, True]

W_xh x_t for all 6 steps at once: torch.Size([6, 3])


- Changing the first token changes **every** hidden state: each one is built from the one before.
- Changing the last token changes only the last one. Earlier states never see later words.
- So $h_6$ needs $h_5$, which needs $h_4$, and so on: six steps in a row.
- Only the input part $W_{xh} x_t$ can be done for all steps at once. The memory part $W_{hh} h_{t-1}$ has to wait.

#### 4.4 It cannot: keep the first word

- Section 3.2 said the training signal shrinks as it travels back through the steps.
- Measure it: how much does the last hidden state react to a change in the first token?
- We use an untrained `nn.RNN` with a larger memory ($H = 16$) and random weights. We repeat it on 8 random networks and take the middle value.

In [5]:
def reaction_to_first_token(model, T):
    X = torch.randn(1, T, 4, requires_grad=True)
    out, _ = model(X)
    out[0, -1].sum().backward()          # a made-up loss that depends on the last step only
    return X.grad[0, 0].norm().item()    # how much it reacts to the FIRST token

def median_over_networks(make_model, T, n=8):
    values = []
    for seed in range(n):
        torch.manual_seed(seed)
        values.append(reaction_to_first_token(make_model(), T))
    return torch.tensor(values).median().item()

make_rnn = lambda: nn.RNN(4, 16, batch_first=True)

print("How much the last step reacts to the first token (RNN):")
for T in (2, 5, 10, 20, 40, 80):
    print(f"  sequence length {T:>2}:  {median_over_networks(make_rnn, T):.1e}")

How much the last step reacts to the first token (RNN):
  sequence length  2:  3.9e-01
  sequence length  5:  4.7e-02
  sequence length 10:  2.0e-03
  sequence length 20:  5.2e-06
  sequence length 40:  5.8e-11


  sequence length 80:  1.3e-21


- The reaction shrinks fast: $0.39$ at 2 tokens, $0.002$ at 10, $0.000005$ at 20.
- At 80 tokens it is about $10^{-21}$. The last step cannot tell what the first token was.
- Each extra step multiplies the reaction by about $0.55$ here: the same shrinking as the 0.8 in 3.2, with real numbers.
- This is the **vanishing gradient**: a change in the first word barely reaches the last step, so the RNN cannot learn from it.
- The weights are random and untrained, so the exact numbers change from network to network. The pattern is what matters.

#### 4.5 The LSTM: does it help?

- Section 3.3 said the LSTM can keep a note by leaving its forget gate open.
- Run the same measurement on `nn.LSTM`, with two settings: PyTorch's default, and forget gates that start mostly open.

In [6]:
make_lstm = lambda: nn.LSTM(4, 16, batch_first=True)

def make_lstm_open():
    m = nn.LSTM(4, 16, batch_first=True)
    with torch.no_grad():
        m.bias_ih_l0[16:32].fill_(1.0)   # forget-gate bias = 1, so the gates start around 0.7 (open)
        m.bias_hh_l0[16:32].zero_()
    return m

print("How much the last step reacts to the first token:")
print(f"{'length':>8} {'RNN':>10} {'LSTM':>10} {'LSTM, open gates':>18}")
for T in (2, 5, 10, 20, 40, 80):
    r = median_over_networks(make_rnn, T)
    l = median_over_networks(make_lstm, T)
    o = median_over_networks(make_lstm_open, T)
    print(f"{T:>8} {r:>10.1e} {l:>10.1e} {o:>18.1e}")

How much the last step reacts to the first token:
  length        RNN       LSTM   LSTM, open gates


       2    3.9e-01    1.3e-01            1.8e-01


       5    4.7e-02    1.8e-02            7.3e-02
      10    2.0e-03    2.4e-03            2.5e-02


      20    5.2e-06    3.0e-05            4.2e-03


      40    5.8e-11    1.4e-09            1.4e-04


      80    1.3e-21    4.7e-17            8.0e-08


- With PyTorch's default setup, the LSTM is only a little better than the RNN: about $0.63$ per step, against $0.55$. It still vanishes.
- With forget gates that start open, it is much better: about $0.83$ per step. At 20 tokens the reaction is $0.004$, about 800 times the RNN's. At 80 tokens it is $8 \times 10^{-8}$, against $10^{-21}$.
- It still shrinks, just more slowly. The LSTM is a fix, not a cure.
- The LSTM helps because its forget gates **can** stay open. These networks are untrained, so we opened the gates by hand, a common trick. In training, the model learns which gates to keep open (3.3).